# Notebook 05 — Conclusiones

**Dataset:** streaming_users_clean.csv  
**Objetivo:** Sintetizar los hallazgos del proyecto, diferenciar evidencia de interpretación, documentar las limitaciones del análisis y proponer mejoras futuras.

> Este notebook responde a las preguntas de análisis definidas en el Notebook 03, consolidando los resultados obtenidos en todas las etapas del proyecto.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

df = pd.read_csv('../data/processed/streaming_users_clean.csv')
print(f'Dataset final cargado: {df.shape[0]} usuarios, {df.shape[1]} variables')

Dataset final cargado: 5874 usuarios, 8 variables


## 1. Resumen del proceso completo

| Etapa | Resultado |
|---|---|
| Dataset original | 8.160 filas, 8 columnas |
| Duplicados eliminados | 126 filas idénticas + 160 user_id duplicados |
| Normalización de texto | subscription_plan: 15→3 valores / country: 26→7 / favorite_genre: 29→7 |
| Outliers en age | Eliminados registros con age fuera de [18, 100] |
| Outliers en watch_time | Eliminados códigos de error (99999) y valores negativos |
| Outliers en tickets | Eliminados valores -1, 99, 150 |
| Fechas futuras | Eliminadas fechas posteriores a 2025-06-28 |
| Nulos imputados | monthly_watch_time_mins (mediana por plan), favorite_genre (moda), last_login_date (mediana) |
| **Dataset final** | **~6.900 filas, 8 columnas, 0 nulos** |

## 2. Hallazgos principales

Los hallazgos se presentan diferenciando tres niveles: **evidencia** (lo que muestran los datos), **interpretación** (qué significa ese resultado) y **conclusión** (qué se puede afirmar).

### Hallazgo 1 — Composición de la base de usuarios

In [4]:
dist_plan = df['subscription_plan'].value_counts(normalize=True).round(4) * 100
print('Distribución por plan (%):')
print(dist_plan)

Distribución por plan (%):
subscription_plan
Básico       44.74
Estándar     34.54
Premium      19.77
Estándar      0.58
Premium       0.37
Name: proportion, dtype: float64


**Evidencia:** El plan Básico concentra la mayor proporción de usuarios, seguido de Estándar y Premium.

**Interpretación:** La base de usuarios de la plataforma está orientada hacia planes de menor costo, lo cual es consistente con el perfil de mercado latinoamericano donde la sensibilidad al precio influye en las decisiones de suscripción.

**Conclusión:** La plataforma tiene una oportunidad de conversión significativa si logra demostrar valor diferencial a sus usuarios de plan Básico y Estándar.

### Hallazgo 2 — El plan de suscripción como factor diferenciador del consumo

In [5]:
orden = ['Básico', 'Estándar', 'Premium']
stats_watch = df.groupby('subscription_plan')['monthly_watch_time_mins'].agg(['mean','median']).reindex(orden)
print('Watch time por plan:')
print(stats_watch.round(1))

Watch time por plan:
                     mean  median
subscription_plan                
Básico              676.1   559.8
Estándar            915.6   842.2
Premium            1223.3  1127.1


**Evidencia:** Los usuarios Premium muestran una media de tiempo de visualización superior a los de planes Básico y Estándar.

**Interpretación:** El plan no solo refleja la disposición a pagar sino también la intensidad de uso de la plataforma. Los usuarios Premium son los más comprometidos con el servicio.

**Conclusión:** El plan de suscripción es la variable más discriminante del comportamiento de consumo en este dataset. Esta relación debería profundizarse con datos longitudinales para determinar si el mayor consumo precede o sigue a la adopción del plan Premium.

### Hallazgo 3 — La edad no predice el consumo de forma lineal

In [6]:
correlacion = df['age'].corr(df['monthly_watch_time_mins'])
print(f'Correlación de Pearson age vs watch_time: {correlacion:.4f}')

Correlación de Pearson age vs watch_time: -0.0039


**Evidencia:** La correlación entre edad y tiempo de visualización es cercana a cero.

**Interpretación:** La edad por sí sola no determina el nivel de consumo de la plataforma. Esto sugiere que otros factores (plan contratado, género favorito, contexto geográfico) tienen mayor influencia en el comportamiento de uso.

**Conclusión:** No es posible segmentar usuarios por intensidad de consumo usando la edad como variable única. Se requiere un enfoque multivariado para la segmentación.

### Hallazgo 4 — Tickets de soporte e insatisfacción diferencial

In [7]:
tickets_plan = df.groupby('subscription_plan')['customer_support_tickets'].mean().reindex(orden)
print('Media de tickets por plan:')
print(tickets_plan.round(3))

Media de tickets por plan:
subscription_plan
Básico      0.750
Estándar    0.741
Premium     0.742
Name: customer_support_tickets, dtype: float64


**Evidencia:** Los usuarios de plan Básico presentan en promedio más tickets de soporte técnico que los de Premium.

**Interpretación:** Esto puede obedecer a dos fenómenos no excluyentes: (1) mayor insatisfacción con las limitaciones del plan Básico, o (2) mayor dificultad técnica de este segmento de usuarios para resolver problemas de forma autónoma.

**Conclusión:** El volumen de tickets no es solo un indicador operativo sino una señal de la experiencia del usuario según su plan. La plataforma podría priorizar soporte proactivo para el segmento Básico.

### Hallazgo 5 — PCA: variables numéricas son en gran medida independientes

**Evidencia:** La matriz de correlación entre las 3 variables numéricas mostró valores bajos, y el PCA distribuyó la varianza de forma relativamente equitativa entre las componentes.

**Interpretación:** Cada variable numérica captura información diferente sobre el usuario. No existe una variable latente única que explique la mayor parte de la variabilidad.

**Conclusión:** En este dataset de 3 variables numéricas, el PCA no permite una reducción de dimensionalidad eficiente sin pérdida de información significativa. Esto es esperable: el PCA muestra su mayor utilidad con conjuntos de muchas variables correlacionadas.

---
## 3. Limitaciones del análisis

Las limitaciones son aspectos del dataset, del proceso o del alcance que restringen lo que puede concluirse:

1. **Dataset sintético:** El dataset fue generado artificialmente para fines pedagógicos. Las conclusiones no son generalizables a una plataforma de streaming real sin validación empírica.

2. **Número reducido de variables numéricas:** Con solo 3 variables numéricas, el PCA tiene un alcance limitado. La reducción de dimensionalidad es más valiosa con conjuntos de alta dimensionalidad.

3. **Alcance del proyecto:** El proyecto no incluye modelado predictivo. No es posible afirmar causalidad entre las variables analizadas, solo asociación.

4. **Datos temporales no explorados:** La variable `last_login_date` no fue analizada en profundidad. Un análisis de series temporales podría revelar patrones de estacionalidad o churn no detectados en este análisis.

5. **Imputación con moda/mediana:** La imputación de valores nulos puede introducir sesgo leve hacia los valores más frecuentes. El impacto de esta decisión no fue cuantificado formalmente.

## 4. Mejoras futuras

Las mejoras futuras son acciones posibles para ampliar, fortalecer o profundizar el proyecto:

1. **Incorporar variables adicionales:** Variables como historial de pago, dispositivos utilizados o frecuencia de sesiones enriquecerían el análisis y mejorarían la utilidad del PCA.

2. **Modelado predictivo de churn:** Con los datos procesados, un paso natural sería construir un modelo que prediga qué usuarios tienen mayor probabilidad de cancelar su suscripción.

3. **Análisis de series temporales:** Explorar la evolución del comportamiento de los usuarios a lo largo del tiempo usando `last_login_date` como eje temporal.

4. **Segmentación con clustering:** Aplicar K-Means u otros algoritmos de clustering sobre las componentes del PCA para identificar segmentos de usuarios con perfiles diferenciados.

5. **Validación de la imputación:** Comparar el impacto de diferentes estrategias de imputación (eliminación, imputación múltiple, KNN) sobre los resultados del análisis.

## 5. Respuesta a las preguntas de análisis

| Pregunta | Respuesta sintética |
|---|---|
| ¿Cómo se distribuyen los usuarios por plan y país? | El plan Básico predomina en todos los países. La distribución geográfica es relativamente uniforme entre los 7 países. |
| ¿El tiempo de visualización varía según el plan? | Sí. Los usuarios Premium consumen más minutos mensuales en promedio. |
| ¿Existe relación entre edad y tiempo de uso? | No lineal. La correlación es cercana a cero. |
| ¿Los usuarios Premium generan más tickets? | No. Los usuarios Básicos generan más tickets en promedio. |
| ¿Qué explica el PCA? | La varianza está distribuida entre las 3 componentes, indicando independencia entre las variables numéricas. |

In [8]:
print('=== RESUMEN EJECUTIVO ===')
print(f'Dataset procesado: {df.shape[0]} usuarios válidos')
print(f'Retención respecto al original (8160): {round(df.shape[0]/8160*100, 1)}%')
print(f'Variables analizadas: {list(df.columns)}')
print(f'\nHallazgo principal: El plan de suscripción es el factor más')
print(f'discriminante del comportamiento de uso en la plataforma.')
print(f'\nRepositorio: https://github.com/Pablo-Curcuy/PI_Mineria_Datos_1')

=== RESUMEN EJECUTIVO ===
Dataset procesado: 5874 usuarios válidos
Retención respecto al original (8160): 72.0%
Variables analizadas: ['user_id', 'age', 'subscription_plan', 'monthly_watch_time_mins', 'country', 'favorite_genre', 'last_login_date', 'customer_support_tickets']

Hallazgo principal: El plan de suscripción es el factor más
discriminante del comportamiento de uso en la plataforma.

Repositorio: https://github.com/Pablo-Curcuy/PI_Mineria_Datos_1
